# 파이프라인 V2: 공정 단위 Split → 베이스라인 선정 → 인코넬 학습 → 몰리브덴 파인튜닝

## 데이터 소스
- **원본 (NAS)**: `Z:\한경훈\인코넬 데이터\●공정&시간 반영 이미지\Dataset_extracted`
- **Split 저장 (NAS)**: `Z:\한경훈\몰리브덴 데이터\●공정&시점 반영 이미지\라벨링\split`

## 공정(test_name) 단위 Split 배정
| Split | 공정 수 | 공정 목록 |
|---|---|---|
| train (24개) | label0: 12개, label1: 12개 | Test1-5,7,8,10,12,28-30 + Test6,31-40,42 |
| val (9개) | label0: 7개, label1: 2개 | Val1-Val9 |
| test (6개) | label0: 3개, label1: 3개 | Test9,11,44 + Test41,43,45 |

> 같은 공정(test_name)의 이미지는 반드시 같은 split에만 포함 (데이터 누수 방지)
> label 2 제외, label 0(정상) / label 1(이상) 만 사용

## 파이프라인
| 단계 | 내용 |
|---|---|
| STEP 0 | 공정 단위 데이터 분할 (NAS → NAS, 원본 보존) |
| STEP 1 | 베이스라인 모델 선정 (ResNet50 / ResNet101 / DenseNet121 / EfficientNet-B0) |
| STEP 2 | 선정 아키텍처 × 3동결전략 → 모델 A 확정 |
| STEP 3 | 모델 A → 몰리브덴 파인튜닝 × 3동결전략 → 모델 B 선정 |
| STEP 4 | 인코넬 테스트셋 A vs B 최종 비교 (파국적 망각 검증) |

In [ ]:
import os, time, copy, random, gc, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 한글 폰트
font_path = r'C:\Windows\Fonts\malgun.ttf'
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

디바이스: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 31.8 GB


c:\Users\msi\anaconda3\envs\ai_project\lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [ ]:
import re

# ── NAS 경로 - 인코넬 ──────────────────────────────────────────
INCONEL_SRC = r'Z:\한경훈\인코넬 데이터\●공정&시간 반영 이미지\Dataset_extracted'
INCONEL_DIR = r'Z:\한경훈\인코넬 데이터\●공정&시간 반영 이미지\Dataset_extracted\split'

# ── NAS 경로 - 몰리브덴 ────────────────────────────────────────
MOLY_SRC    = r'Z:\한경훈\몰리브덴 데이터\●공정&시점 반영 이미지\라벨링'
MOLY_DIR    = r'Z:\한경훈\몰리브덴 데이터\●공정&시점 반영 이미지\라벨링\split'

SAVE_DIR    = r'C:\Users\msi\Desktop\팀프로젝트\인공지능\code\main\checkpoints_pipeline_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 인코넬: 공정 prefix 기준 Split ───────────────────────────
# 파일명 예: Test1_160A_20TS_160WFR_t0.12s.png → prefix: Test1
INCONEL_TRAIN_PROCS = {
    'Test1', 'Test2', 'Test3', 'Test4', 'Test5', 'Test7', 'Test8',
    'Test10', 'Test12', 'Test28', 'Test29', 'Test30',             # label0 공정
    'Test6', 'Test31', 'Test32', 'Test33', 'Test34', 'Test35',
    'Test36', 'Test37', 'Test38', 'Test39', 'Test40', 'Test42',   # label1 공정
}
INCONEL_VAL_PROCS  = {'Val1', 'Val2', 'Val3', 'Val4', 'Val5', 'Val6', 'Val7', 'Val8', 'Val9'}
INCONEL_TEST_PROCS = {'Test9', 'Test11', 'Test44', 'Test41', 'Test43', 'Test45'}

# ── 몰리브덴: 공정 번호 기준 Split ───────────────────────────
# 파일명 예: 04._STO3_..._t17.92s.png → 번호: 4
# Val  : 38~50번 (Val 계열 공정)
# Test : 4, 35, 36, 37번 (이상 포함 STO + 인접 정상 STO)
# Train: 나머지 (STO 01~34 + BK 51~70)
MOLY_VAL_NUMS  = set(range(38, 51))
MOLY_TEST_NUMS = {4, 35, 36, 37}

# ── 공통 하이퍼파라미터 ───────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42
CLASS_NAMES = ['정상', '이상']

# ── Early Stopping ────────────────────────────────────────────
MAX_EPOCHS = 30
PATIENCE   = 7

# ── 베이스라인 비교 아키텍처 ──────────────────────────────────
BASELINE_ARCHS = ['resnet50', 'resnet101', 'densenet121', 'efficientnet_b0']
SELECTED_ARCH  = None

# ── 학습률 ────────────────────────────────────────────────────
IN_LR   = 1e-4
MOLY_LR = 1e-5

FREEZE_RATIOS = [0.0, 0.5, 0.85]
FREEZE_LABELS = ['전략1(0%동결)', '전략2(50%동결)', '전략3(85%동결)']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('설정 완료')
print(f'  인코넬 Train:{len(INCONEL_TRAIN_PROCS)}공정 / Val:{len(INCONEL_VAL_PROCS)}공정 / Test:{len(INCONEL_TEST_PROCS)}공정')
print(f'  몰리브덴 Val:38~50번 / Test:{{4,35,36,37}}번 / Train:나머지')

설정 완료
  인코넬 Train:24공정 / Val:9공정 / Test:6공정
  몰리브덴 Val:38~50번 / Test:{4,35,36,37}번 / Train:나머지


## STEP 0. 데이터 분할 (영상 단위, 최초 1회만 실행)

### NAS에 있는 원본 데이터 split

In [5]:
# ── STEP 0-A: 인코넬 공정 단위 Split (실제 파일 기준, 원본 보존) ─
inconel_split_dirs = [os.path.join(INCONEL_DIR, s, c)
                      for s in ['train', 'val', 'test'] for c in ['0', '1']]

if all(os.path.exists(d) and len(os.listdir(d)) > 0 for d in inconel_split_dirs):
    total = sum(len(os.listdir(d)) for d in inconel_split_dirs)
    print(f'[인코넬] 분할 폴더 이미 존재 ({total}개) → 스킵')
else:
    print('[인코넬] 공정 단위 분할 시작...')
    for d in inconel_split_dirs:
        os.makedirs(d, exist_ok=True)

    def get_inconel_split(fname):
        process = re.sub(r'_t[\d.]+s\.png$', '', fname)
        prefix = process.split('_')[0]
        if prefix in INCONEL_TRAIN_PROCS: return 'train'
        if prefix in INCONEL_VAL_PROCS:   return 'val'
        if prefix in INCONEL_TEST_PROCS:  return 'test'
        return None

    stats = {'train': {0:0, 1:0}, 'val': {0:0, 1:0}, 'test': {0:0, 1:0}}
    skipped = 0
    for cls in ['0', '1']:
        for fname in os.listdir(os.path.join(INCONEL_SRC, cls)):
            split = get_inconel_split(fname)
            if split is None:
                skipped += 1
                continue
            shutil.copy2(os.path.join(INCONEL_SRC, cls, fname),
                         os.path.join(INCONEL_DIR, split, cls, fname))
            stats[split][int(cls)] += 1

    print(f'\n{"Split":<8} {"정상(0)":>10} {"이상(1)":>10} {"합계":>8}')
    print('-' * 40)
    for sp in ['train', 'val', 'test']:
        s = stats[sp]
        print(f'{sp:<8} {s[0]:>10} {s[1]:>10} {s[0]+s[1]:>8}')
    if skipped: print(f'미배정: {skipped}개')
    print('[인코넬] 분할 완료!')

[인코넬] 공정 단위 분할 시작...


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'Z:\\'

In [ ]:
# ── STEP 0-B: 몰리브덴 공정 단위 Split (실제 파일 기준, 원본 보존) ─
moly_split_dirs = [os.path.join(MOLY_DIR, s, c)
                   for s in ['train', 'val', 'test'] for c in ['0', '1']]

if all(os.path.exists(d) and len(os.listdir(d)) > 0 for d in moly_split_dirs):
    total = sum(len(os.listdir(d)) for d in moly_split_dirs)
    print(f'[몰리브덴] 분할 폴더 이미 존재 ({total}개) → 스킵')
else:
    print('[몰리브덴] 공정 단위 분할 시작...')
    for d in moly_split_dirs:
        os.makedirs(d, exist_ok=True)

    def get_moly_split(fname):
        process = re.sub(r'_t[\d.]+s\.png$', '', fname)
        num = int(process.split('.')[0])
        if num in MOLY_VAL_NUMS:  return 'val'
        if num in MOLY_TEST_NUMS: return 'test'
        return 'train'

    stats = {'train': {0:0, 1:0}, 'val': {0:0, 1:0}, 'test': {0:0, 1:0}}
    for cls in ['0', '1']:
        for fname in os.listdir(os.path.join(MOLY_SRC, cls)):
            split = get_moly_split(fname)
            shutil.copy2(os.path.join(MOLY_SRC, cls, fname),
                         os.path.join(MOLY_DIR, split, cls, fname))
            stats[split][int(cls)] += 1

    print(f'\n{"Split":<8} {"정상(0)":>10} {"이상(1)":>10} {"합계":>8}')
    print('-' * 40)
    for sp in ['train', 'val', 'test']:
        s = stats[sp]
        print(f'{sp:<8} {s[0]:>10} {s[1]:>10} {s[0]+s[1]:>8}')
    print('[몰리브덴] 분할 완료!')